# Whisper-small (frozen) + SVM

**Модель:** Whisper-small encoder (все веса заморожены) → mean+std+max pooling → SVM RBF

Один эксперимент, одна модель. 5-fold Stratified CV на train+val, финальная оценка на holdout test.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from transformers import WhisperProcessor, WhisperModel

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds, get_durations, load_audio
from shared.results_utils import save_result_csv

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

MODEL_ID    = "openai/whisper-small"
EXP_ID      = "exp_whisper_svm"
EXP_NAME    = "Whisper-small encoder + SVM"
MODEL_LABEL = "Whisper-small (frozen) + SVM RBF"

print(f"Device: {DEVICE}")
print(f"Model:  {MODEL_ID}")

In [ ]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()

print(f"Train+Val: {len(paths_trainval)}  |  Test: {len(paths_test)}")
print(f"Bad (train+val): {sum(l == config.CLASS_BAD for l in labels_trainval)}")
print(f"Bad (test):      {sum(l == config.CLASS_BAD for l in labels_test)}")

In [ ]:
# Загрузка Whisper-small, извлечение эмбеддингов
processor = WhisperProcessor.from_pretrained(MODEL_ID)
whisper   = WhisperModel.from_pretrained(MODEL_ID).to(DEVICE)
whisper.eval()

EMBED_DIM = whisper.config.d_model  # 768 для small
print(f"d_model = {EMBED_DIM}  →  итоговый вектор: {EMBED_DIM * 3} (mean+std+max)")

def extract_fn(path):
    y, _ = load_audio(path, sr=16000)
    feats = processor.feature_extractor(y, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        hs = whisper.encoder(
            input_features=feats.input_features.to(DEVICE)
        ).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

print("Извлечение признаков train+val...")
t0 = time.perf_counter()
X_embeds_trainval = build_feature_matrix(paths_trainval, extract_fn, n_jobs=1)
print(f"  готово за {time.perf_counter() - t0:.1f} с")

print("Извлечение признаков test...")
t0 = time.perf_counter()
X_embeds_test = build_feature_matrix(paths_test, extract_fn, n_jobs=1)
print(f"  готово за {time.perf_counter() - t0:.1f} с")

del whisper
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Добавляем длительность и letter-признаки
dur_trainval = get_durations(paths_trainval).reshape(-1, 1)
dur_test     = get_durations(paths_test).reshape(-1, 1)
X_trainval = np.hstack([X_embeds_trainval, letters_trainval, dur_trainval])
X_test     = np.hstack([X_embeds_test,     letters_test,     dur_test])

print(f"\nX_trainval shape: {X_trainval.shape}")
print(f"X_test     shape: {X_test.shape}")

In [ ]:
# 5-fold Stratified CV
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}

path_to_idx = {p: i for i, p in enumerate(paths_trainval)}
fold_results = []
t0 = time.perf_counter()

for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
        get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

    print(f"\nФолд {fold+1}/{config.CV_N_SPLITS}")

    idx_tr  = [path_to_idx[p] for p in paths_tr]
    idx_val = [path_to_idx[p] for p in paths_val]
    X_tr  = np.hstack([
        X_embeds_trainval[idx_tr],
        letters_tr,
        get_durations(paths_tr).reshape(-1, 1),
    ])
    X_val = np.hstack([
        X_embeds_trainval[idx_val],
        letters_val,
        get_durations(paths_val).reshape(-1, 1),
    ])

    grid = GridSearchCV(pipeline, param_grid, cv=3,
                        scoring="f1_macro", refit=True, n_jobs=-1)
    grid.fit(X_tr, labels_tr)
    clf = grid.best_estimator_
    print(f"  best={grid.best_params_}")

    y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
    thr = find_optimal_threshold(labels_val, y_proba_val)
    metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=True)
    fold_results.append(metrics)

train_time_sec = time.perf_counter() - t0
cv_agg = evaluate_cv_folds(fold_results)
print_cv_summary(cv_agg)

df_folds = pd.DataFrame(fold_results)
df_folds.index = [f"Fold {i+1}" for i in range(len(fold_results))]
display(df_folds[["accuracy", "f1_macro", "f1_bad", "roc_auc", "threshold"]])

In [ ]:
# Финальная модель на train+val → оценка на holdout test
print("Финальная модель на train+val...")
final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                          scoring="f1_macro", refit=True, n_jobs=-1)
final_grid.fit(X_trainval, labels_trainval)
print(f"  best_params={final_grid.best_params_}")

cv_threshold = cv_agg["threshold_mean"]
y_proba_test = final_grid.best_estimator_.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)

# Сохраняем предсказания
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= cv_threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id=EXP_ID,
    experiment_name=EXP_NAME,
    model=MODEL_LABEL,
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    cv_f1_bad_std=cv_agg["f1_bad_std"],
    cv_f1_macro_std=cv_agg["f1_macro_std"],
    cv_accuracy_std=cv_agg["accuracy_std"],
    cv_roc_auc_std=cv_agg["roc_auc_std"],
    cv_precision_bad_std=cv_agg["precision_bad_std"],
    cv_recall_bad_std=cv_agg["recall_bad_std"],
    cv_threshold_std=cv_agg["threshold_std"],
    embed_dim=EMBED_DIM * 3,
    embed_dim_note=f"Whisper-small {EMBED_DIM}-dim × 3 (mean+std+max)",
    notes=f"5-fold CV + holdout | best={final_grid.best_params_} | thr={cv_threshold:.2f}",
    train_time_sec=train_time_sec,
    append=False,
)

print("\nИтоговые результаты:")
display(pd.read_csv(exp_dir / "result.csv"))